In [ ]:
import requests
import pandas as pd
import time

In [ ]:
# Trocar Futuramente por consulta na Wiki + snake case format
bond_mods_list = [
    "aerial_bond",
    "astral_bond",
    "contagious_bond",
    "covert_bond",
    "duplex_bond",
    "manifold_bond",
    "momentous_bond",
    "mystic_bond",
    "reinforced_bond",
    "restorative_bond",
    "seismic_bond",
    "tandem_bond",
    "tenacious_bond",
    "vicious_bond"
]  

In [ ]:
def get_order_request(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()
    return response

def get_bond_mod_offers(bond_mods_list: list[str]):
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting orders from {mod_name}")
        url = f"https://api.warframe.market/v2/orders/item/{mod_name}"
        mod_orders = get_order_request(url)
        df_order = pd.DataFrame(mod_orders.json()["data"])
        df_concat = pd.concat([df_concat, df_order])
    df_concat['user'] = df_concat['user'].str['ingameName']
    df_concat = df_concat[['user', 'rank', 'platinum', 'type', 'quantity', 'perTrade', 'createdAt', 'updatedAt']]
    df_concat = df_concat.sort_values('rank', ascending=False)
    return df_concat

def get_bond_mod_timeseries(bond_mods_list:list[str]):
    collect_modes = ["statistics_closed", "statistics_live"]
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting timeseries from {mod_name}")
        url = f"https://api.warframe.market/v1/items/{mod_name}/statistics"
        mod_orders = get_order_request(url)
        for mode in collect_modes:
            df_order = pd.DataFrame(mod_orders.json()["payload"][mode]["90days"])
            df_order["mode"] = ("live" if mode == "statistics_live" else "closed")
            df_concat = pd.concat([df_concat, df_order])
    df_concat = df_concat.sort_values('datetime', ascending=False)
    return df_concat

In [ ]:
orders_df = get_bond_mod_offers(bond_mods_list)
display(orders_df)

In [ ]:
orders_df.to_csv('../data/bond_orders.csv', sep=';')

In [ ]:
timeseries_df = get_bond_mod_timeseries(bond_mods_list)
display(timeseries_df)

In [ ]:
timeseries_df.to_csv('../data/bond_90days_orders.csv', sep=';')